# Unit 4 (CO-4): Multi-Agent Systems and Collaboration
**Shared task:** Plan a one-day technical workshop for college students (topic, schedule, resources, budget) within **Rs 8,000**.

**Agents (3):**
| Agent | Role | Tools |
|---|---|---|
| Topic Researcher | suggests the workshop topic | topic checklist |
| Budget Planner | estimates the cost | budget sheet, calculator |
| Workshop Coordinator | passes work between agents, checks the budget, writes the final plan | calculator, budget check, decision rules |

**Framework:** LangGraph. The Coordinator is the supervisor node and the other two agents always report back to it.

**Runs fully locally:** no API key and no internet needed (only `pip install langgraph` once). The agents here are **rule-based**: each agent's "thinking" is a normal Python function, marked `# brain`. These are the places where an LLM call could be plugged in later. The collaboration part (roles, hand-offs, negotiation, checks) is real and is what this notebook demonstrates.

All prices are **sample estimates assumed for the demo**, not real quotes.

## 1. Setup

In [1]:
%pip install -q langgraph

Note: you may need to restart the kernel to use updated packages.


In [2]:
from typing import TypedDict, Optional
from langgraph.graph import StateGraph, START, END

## 2. Tools
Plain Python functions. The numbers come from code.

In [3]:
BUDGET_LIMIT = 8000   # Rs, set by the shared task

# Topic Researcher's tool: small checklist of candidate topics (my own assumptions)
TOPIC_CHECKLIST = {
    "Generative AI and prompt writing": {"needs_laptop": True,  "free_tools": True,  "beginner_friendly": True},
    "Web development with HTML/CSS":    {"needs_laptop": True,  "free_tools": True,  "beginner_friendly": True},
    "Cloud computing basics":           {"needs_laptop": True,  "free_tools": False, "beginner_friendly": False},
    "Robotics with Arduino kits":       {"needs_laptop": False, "free_tools": False, "beginner_friendly": True},
}

# Budget Planner's tool: budget sheet. "normal" = usual price, "lowest" = price if college resources are used.
BUDGET_SHEET = {
    "venue_and_setup": {"normal": 3000, "lowest": 3000},
    "equipment":       {"normal": 2000, "lowest": 2000},
    "refreshments":    {"normal": 2500, "lowest": 1800},
    "materials":       {"normal": 1500, "lowest": 1200},
}

def topic_checklist():
    return TOPIC_CHECKLIST

def calculator(items: dict) -> int:
    """Adds up the cost items. Ignores anything that is not in the budget sheet."""
    return sum(int(v) for k, v in items.items() if k in BUDGET_SHEET)

def budget_check(total: int, limit: int = BUDGET_LIMIT) -> dict:
    return {"total": total, "limit": limit, "within_budget": total <= limit, "difference": total - limit}

## 3. Agents and shared state
Every agent reads and writes the same `state`. The Coordinator decides who goes next.

In [4]:
class State(TypedDict):
    task: str
    topic: Optional[dict]        # from Topic Researcher
    free_tools_note: Optional[str]
    estimate: Optional[dict]     # from Budget Planner
    feedback: Optional[str]      # Coordinator's message to Budget Planner
    over_by: int                 # how much the last estimate was above the limit
    rounds: int                  # how many times the estimate was sent back
    final_plan: Optional[str]
    next: str                    # who gets control next
    log: list                    # conversation log for the report

MAX_ROUNDS = 3

def say(state, who, msg):
    line = f"[{who}] {msg}"
    print(line, "\n")
    state["log"].append(line)

# ---------------- Agent 1: Topic Researcher ----------------
# Rule: never invent confirmed speakers, registrations or facts about student demand. Does no budgeting.
def topic_researcher(state: State):
    checklist = topic_checklist()
    if state["topic"] is None:
        # brain: pick the first topic that is beginner friendly, needs a laptop and has free tools
        best = next(name for name, f in checklist.items() if f["beginner_friendly"] and f["free_tools"] and f["needs_laptop"])
        state["topic"] = {"topic": best,
                          "format": "short introduction, demonstrations and a hands-on activity",
                          "why": "assumed to suit beginners (no student-demand data was used)"}
        say(state, "Topic Researcher",
            f"I suggest a beginner-friendly workshop on {best}. It can include a {state['topic']['format']}. ({state['topic']['why']})")
    else:
        # brain: answer the Coordinator's question about free tools using the checklist
        f = checklist[state["topic"]["topic"]]
        if f["free_tools"] and f["needs_laptop"]:
            note = "The practical activity can still run with free tools and participants' own laptops, so the learning part does not need to be removed."
        else:
            note = "This topic needs paid tools or kits, so cutting cost may remove part of the hands-on activity."
        state["free_tools_note"] = note
        say(state, "Topic Researcher", note)
    state["next"] = "coordinator"
    return state

# ---------------- Agent 2: Budget Planner ----------------
# Rules: numbers are ESTIMATES, never confirmed prices. Never go below the 'lowest' value. Does not choose the topic.
def budget_planner(state: State):
    if state["feedback"] is None:
        # brain: first estimate uses the normal price for every category
        items = {k: v["normal"] for k, v in BUDGET_SHEET.items()}
        note = "First estimate using normal prices."
    else:
        # brain: cut flexible categories (normal > lowest) until the over_by amount is covered, never below 'lowest'
        items = {k: v["normal"] for k, v in BUDGET_SHEET.items()}
        need = state["over_by"]
        for k, v in sorted(BUDGET_SHEET.items(), key=lambda kv: kv[1]["normal"] - kv[1]["lowest"], reverse=True):
            cut = min(need, v["normal"] - v["lowest"])
            items[k] -= cut
            need -= cut
        note = "Revised estimate using available college resources where possible."
    state["estimate"] = {"items": items, "claimed_total": sum(items.values()), "note": note}
    say(state, "Budget Planner", f"Estimate: {items} | I think the total is Rs {sum(items.values())}. {note}")
    state["next"] = "coordinator"
    return state

# ---------------- Agent 3: Workshop Coordinator (supervisor) ----------------
# Rules: never ignore missing information and never accept conflicting suggestions without checking them.
def write_final_plan(state, items, total):
    t = state["topic"]
    return (f"Topic: {t['topic']}\n"
            f"Format: {t['format']}.\n"
            "Schedule: 10:00 Introduction | 11:00 Demonstrations | 13:00 Lunch | 14:00 Hands-on activity | 15:30 Wrap-up and feedback\n"
            "Resources: college seminar hall, projector, free tools, participants' own laptops.\n"
            "Budget (estimates, not confirmed prices): " + ", ".join(f"{k.replace('_', ' ')} Rs {v}" for k, v in items.items())
            + f". Total Rs {total} (limit Rs {BUDGET_LIMIT}).")

def coordinator(state: State):
    # 1) nothing yet -> start the task, ask Topic Researcher
    if state["topic"] is None:
        say(state, "Coordinator", f"Starting. {state['task']} Sending the topic requirement to the Topic Researcher.")
        state["next"] = "topic_researcher"
        return state

    # 2) topic is ready but no estimate -> ask Budget Planner
    if state["estimate"] is None:
        if state["feedback"] is None:
            say(state, "Coordinator", "Topic looks suitable. Passing it to the Budget Planner for resources and cost.")
        else:
            say(state, "Coordinator", "Sending the revision request (with the Topic Researcher's answer) to the Budget Planner.")
        state["next"] = "budget_planner"
        return state

    # 3) estimate came back -> do NOT trust the planner's numbers, check them
    items = state["estimate"]["items"]
    real_total = calculator(items)
    claimed = state["estimate"].get("claimed_total")
    if claimed != real_total:
        say(state, "Coordinator", f"Warning: planner said Rs {claimed} but the calculator says Rs {real_total}. Using the calculator value.")
    too_low = [k for k, v in items.items() if k not in BUDGET_SHEET or v < BUDGET_SHEET[k]["lowest"]]
    check = budget_check(real_total)
    say(state, "Coordinator", f"Budget check -> total Rs {real_total}, limit Rs {BUDGET_LIMIT}, within budget: {check['within_budget']}")

    if too_low:
        reason = f"Unrealistic estimate: {too_low} are below the lowest possible price in the budget sheet."
        state["over_by"] = max(sum(v["normal"] for v in BUDGET_SHEET.values()) - BUDGET_LIMIT, 0)   # real saving needed
    elif check["within_budget"]:
        state["final_plan"] = write_final_plan(state, items, real_total)
        say(state, "Coordinator", "Plan accepted. FINAL PLAN:\n" + state["final_plan"])
        state["next"] = "end"
        return state
    else:
        reason = (f"Total Rs {real_total} is Rs {check['difference']} above the Rs {BUDGET_LIMIT} limit. "
                  "Reduce the cost without removing the main learning activity.")
        state["over_by"] = check["difference"]

    if state["rounds"] >= MAX_ROUNDS:
        state["final_plan"] = f"NOT APPROVED after {MAX_ROUNDS} revisions: {reason} A human should decide."
        say(state, "Coordinator", state["final_plan"])
        state["next"] = "end"
        return state

    # send back. If we have not yet asked the Topic Researcher about free tools, ask first.
    state["rounds"] += 1
    state["feedback"] = reason
    say(state, "Coordinator", reason)
    state["estimate"] = None
    state["next"] = "topic_researcher" if state["free_tools_note"] is None else "budget_planner"
    return state

## 4. Wiring the graph (hand-offs)
Topic Researcher and Budget Planner always report back to the Coordinator; only the Coordinator can finish the run.

In [5]:
def after_coordinator(state: State):
    return END if state["next"] == "end" else state["next"]

def build_app(budget_node=budget_planner):
    g = StateGraph(State)
    g.add_node("coordinator", coordinator)
    g.add_node("topic_researcher", topic_researcher)
    g.add_node("budget_planner", budget_node)
    g.add_edge(START, "coordinator")
    g.add_conditional_edges("coordinator", after_coordinator,
                            {"topic_researcher": "topic_researcher", "budget_planner": "budget_planner", END: END})
    g.add_edge("topic_researcher", "coordinator")
    g.add_edge("budget_planner", "coordinator")
    return g.compile()

app = build_app()

# Diagram of the hand-offs (paste this Mermaid text into mermaid.live if you need a picture for the report)
print(app.get_graph().draw_mermaid())

def new_state():
    return {"task": TASK, "topic": None, "free_tools_note": None, "estimate": None, "feedback": None,
            "over_by": 0, "rounds": 0, "final_plan": None, "next": "", "log": []}

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	coordinator(coordinator)
	topic_researcher(topic_researcher)
	budget_planner(budget_planner)
	__end__([<p>__end__</p>]):::last
	__start__ --> coordinator;
	budget_planner --> coordinator;
	coordinator -.-> __end__;
	coordinator -.-> budget_planner;
	coordinator -.-> topic_researcher;
	topic_researcher --> coordinator;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## 5. One full run

In [6]:
TASK = ("Plan a one-day technical workshop for college students. Suggest a useful topic, check the expected "
        f"resources and keep the estimated budget within Rs {BUDGET_LIMIT:,}.")

result = app.invoke(new_state(), {"recursion_limit": 30})
print("Estimate revisions needed:", result["rounds"])

[Coordinator] Starting. Plan a one-day technical workshop for college students. Suggest a useful topic, check the expected resources and keep the estimated budget within Rs 8,000. Sending the topic requirement to the Topic Researcher. 

[Topic Researcher] I suggest a beginner-friendly workshop on Generative AI and prompt writing. It can include a short introduction, demonstrations and a hands-on activity. (assumed to suit beginners (no student-demand data was used)) 

[Coordinator] Topic looks suitable. Passing it to the Budget Planner for resources and cost. 

[Budget Planner] Estimate: {'venue_and_setup': 3000, 'equipment': 2000, 'refreshments': 2500, 'materials': 1500} | I think the total is Rs 9000. First estimate using normal prices. 

[Coordinator] Budget check -> total Rs 9000, limit Rs 8000, within budget: False 

[Coordinator] Total Rs 9000 is Rs 1000 above the Rs 8000 limit. Reduce the cost without removing the main learning activity. 

[Topic Researcher] The practical activi

## 6. Save the run log (use this in Section 5 of the report)

In [7]:
with open("run_log.txt", "w") as f:
    f.write("\n\n".join(result["log"]))
print("saved run_log.txt with", len(result["log"]), "messages")

saved run_log.txt with 11 messages


## 7. Failure tests: what if the Budget Planner passes on a wrong result?
Two faults are injected on purpose:
1. the planner **lies about its total** (says Rs 7,500 when the items add up to Rs 9,000);
2. the planner **cheats** by dropping equipment and refreshments to Rs 0 to look cheap.

The Coordinator must not accept either. It recomputes the total and checks every item against the budget sheet.

In [8]:
def lying_planner(state):
    state = budget_planner(state)
    state["estimate"]["claimed_total"] = 7500
    print("   >>> fault injected: claimed total changed to Rs 7500\n")
    return state

fault_count = {"n": 0}
def cheating_planner(state):
    state = budget_planner(state)
    if fault_count["n"] == 0:
        state["estimate"]["items"]["equipment"] = 0
        state["estimate"]["items"]["refreshments"] = 0
        state["estimate"]["claimed_total"] = 4500
        print("   >>> fault injected: equipment and refreshments set to Rs 0\n")
    fault_count["n"] += 1
    return state

print("=========== FAULT 1: lying total ===========\n")
r1 = build_app(lying_planner).invoke(new_state(), {"recursion_limit": 30})
print("=========== FAULT 2: unrealistic cheating estimate ===========\n")
r2 = build_app(cheating_planner).invoke(new_state(), {"recursion_limit": 30})

print("---- summary ----")
for name, r in (("lying total", r1), ("cheating estimate", r2)):
    caught = [l for l in r["log"] if "Warning" in l or "Unrealistic" in l]
    print(f"{name}: caught = {bool(caught)}, revisions = {r['rounds']}, final plan approved = {not r['final_plan'].startswith('NOT')}")

=========== FAULT 1: lying total ===========

[Coordinator] Starting. Plan a one-day technical workshop for college students. Suggest a useful topic, check the expected resources and keep the estimated budget within Rs 8,000. Sending the topic requirement to the Topic Researcher. 

[Topic Researcher] I suggest a beginner-friendly workshop on Generative AI and prompt writing. It can include a short introduction, demonstrations and a hands-on activity. (assumed to suit beginners (no student-demand data was used)) 

[Coordinator] Topic looks suitable. Passing it to the Budget Planner for resources and cost. 

[Budget Planner] Estimate: {'venue_and_setup': 3000, 'equipment': 2000, 'refreshments': 2500, 'materials': 1500} | I think the total is Rs 9000. First estimate using normal prices. 

   >>> fault injected: claimed total changed to Rs 7500

[Coordinator] Warning: planner said Rs 7500 but the calculator says Rs 9000. Using the calculator value. 

[Coordinator] Budget check -> total Rs 